In [ ]:
# Run this cell first
!umount /content/drive
!rm -rf /content/drive

from google.colab import drive
drive.mount('/content/drive')
# Mount Google Drive
# drive.mount('/content/drive', force_remount=True)

umount: /content/drive: no mount point specified.
Mounted at /content/drive


# Train Model 2024-12-23


In [7]:
# !pip install mediapipe
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 903.1/903.1 kB 37.7 MB/s eta 0:00:00


In [5]:
# Check if GPU is available
!nvidia-smi

Mon Dec 23 09:59:53 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              12W /  70W |      3MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [1]:
#  Mount Drive and Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
from collections import Counter
from tqdm import tqdm
from google.colab import drive
from ultralytics import YOLO

drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Mounted at /content/drive
Using device: cuda


In [22]:
#  Flow Computer
class FlowComputer:
    def __init__(self):
        self.flow_params = dict(
            pyr_scale=0.5,
            levels=3,
            winsize=15,
            iterations=3,
            poly_n=5,
            poly_sigma=1.2,
            flags=0
        )

    def compute_flow(self, frame1, frame2):
        grey1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
        grey2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

        flow = cv2.calcOpticalFlowFarneback(
            grey1, grey2, None, **self.flow_params)

        mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
        hsv = np.zeros_like(frame1)
        hsv[..., 1] = 255
        hsv[..., 0] = ang * 180 / np.pi / 2
        hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
        flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

        return flow_rgb

In [58]:
#  Dataset Class
class GaelicActionDataset(Dataset):
    def __init__(self, root_dir, yolo_model_path, clips_per_class=None, sequence_length=16, action_classes=None):
        self.root_dir = Path(root_dir)
        self.yolo_model = YOLO(yolo_model_path)  # Load YOLO here
        self.sequence_length = sequence_length
        self.flow_computer = FlowComputer()


        # Count videos per action
        action_counts = {}
        action_clips = {}

        for action_folder in self.root_dir.iterdir():
            if action_folder.is_dir() and (action_classes is None or action_folder.name in action_classes):
                clips = list(action_folder.glob('*.mp4'))
                count = len(clips)
                if count > 0:
                    action_counts[action_folder.name] = count
                    action_clips[action_folder.name] = clips

        # Set clips per class
        if clips_per_class is None:
            clips_per_class = min(action_counts.values())

        print(f"\nUsing {clips_per_class} clips per class")

        # Create balanced dataset
        self.action_map = {action: idx for idx, action in enumerate(action_clips.keys())}
        self.clips = []

        for action, clips in action_clips.items():
            selected_clips = random.sample(clips, clips_per_class)
            self.clips.extend([(clip, self.action_map[action]) for clip in selected_clips])

        random.shuffle(self.clips)

    def __len__(self):
        return len(self.clips)



    def __getitem__(self, idx):
        clip_path, label = self.clips[idx]
        frames = self._load_clip(clip_path)
        processed_sequences = []
        flow_sequences = []

        # Process sequences of frames
        for i in range(len(frames) - 1):
            # Get frame and next frame
            frame = frames[i].astype(np.uint8)
            next_frame = frames[i+1].astype(np.uint8)

            # Process frames
            results = self.yolo_model.predict(frame, verbose=False)
            mask = self._create_detection_mask(frame, results[0].boxes)

            # Get RGB and flow features
            masked_frame = frame * mask
            flow = self.flow_computer.compute_flow(frame, next_frame)
            masked_flow = flow * mask

            # Convert to tensors with correct dimensions
            processed_sequences.append(torch.from_numpy(masked_frame).permute(2, 0, 1))
            flow_sequences.append(torch.from_numpy(masked_flow).permute(2, 0, 1))

        # Stack and concatenate
        rgb_sequence = torch.stack(processed_sequences)  # [T, C, H, W]
        flow_sequence = torch.stack(flow_sequences)      # [T, C, H, W]

        # Combine RGB and flow channels
        sequence = torch.cat([rgb_sequence, flow_sequence], dim=1)  # [T, 2C, H, W]
        sequence = sequence.permute(1, 0, 2, 3)  # [2C, T, H, W]

        return sequence.float() / 255.0, label

    def _create_detection_mask(self, frame, boxes):
        mask = np.zeros_like(frame)
        for box in boxes:
            if box.cls.item() in [0, 1]:  # Player or ball
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                mask[int(y1):int(y2), int(x1):int(x2)] = 1
        return mask


    def _load_clip(self, clip_path):
        frames = []
        cap = cv2.VideoCapture(str(clip_path))

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)

        cap.release()

        # Handle empty or too short videos
        if len(frames) == 0:
            # Create a black frame
            black_frame = np.zeros((224, 224, 3), dtype=np.uint8)
            frames = [black_frame] * self.sequence_length
        elif len(frames) < self.sequence_length:
            # Duplicate last frame to reach required length
            last_frame = frames[-1]
            while len(frames) < self.sequence_length:
                frames.append(last_frame.copy())

        # Sample frames uniformly
        step = max(1, len(frames) // self.sequence_length)
        frames = frames[::step][:self.sequence_length]

        return frames



In [59]:
class ActionRecognitionModel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(6, 64, kernel_size=(3,3,3), padding=(1,1,1)),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(1,2,2)),

            nn.Conv3d(64, 128, kernel_size=(3,3,3), padding=(1,1,1)),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2,2,2)),

            nn.Conv3d(128, 256, kernel_size=(3,3,3), padding=(1,1,1)),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=(2,2,2))
        )

        # Add adaptive pooling to get fixed size
        self.adaptive_pool = nn.AdaptiveAvgPool3d((1, 7, 7))

        self.classifier = nn.Sequential(
            nn.Linear(256 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [60]:
#  Training Function
def train_model(model, train_loader, val_loader, num_epochs=30, device='cuda'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5)

    best_val_acc = 0.0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')

        # Training
        model.train()
        train_loss = 0
        correct = 0
        total = 0

        for sequences, labels in tqdm(train_loader):
            sequences = sequences.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        train_acc = 100. * correct / total
        print(f'Train Loss: {train_loss/len(train_loader):.4f}')
        print(f'Train Accuracy: {train_acc:.2f}%')

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for sequences, labels in tqdm(val_loader):
                sequences = sequences.to(device)
                labels = labels.to(device)

                outputs = model(sequences)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_acc = 100. * correct / total
        print(f'Val Loss: {val_loss/len(val_loader):.4f}')
        print(f'Val Accuracy: {val_acc:.2f}%')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '/content/drive/MyDrive/2024_12_24.pth')

        scheduler.step(val_acc)

In [61]:
#  Main Training
from ultralytics import YOLO

def main():
    random.seed(42)
    torch.manual_seed(42)

    action_classes = ['hand passes', 'foot passes','carries','Tackles']  # Specify desired classes
    yolo_path = ('/content/drive/MyDrive/action_recognition_videos/best.pt')

    train_dataset = GaelicActionDataset(
        root_dir='/content/drive/MyDrive/action_recognition_videos/output/train',
        yolo_model_path=yolo_path,
        clips_per_class=200,
            action_classes=action_classes

    )

    val_dataset = GaelicActionDataset(
        root_dir='/content/drive/MyDrive/action_recognition_videos/output/val',
        yolo_model_path=yolo_path,
        clips_per_class=20,
                    action_classes=action_classes

    )

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,    pin_memory=True,
 num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

    print(f"Training set: {len(train_dataset)} clips")
    print(f"Validation set: {len(val_dataset)} clips")

    model = ActionRecognitionModel(num_classes=len(train_dataset.action_map))
    train_model(model, train_loader, val_loader, device=device)

if __name__ == '__main__':
    import multiprocessing as mp
    try:
        mp.set_start_method('spawn')
    except RuntimeError:
        pass
    main()


Using 200 clips per class

Using 20 clips per class
Training set: 800 clips
Validation set: 80 clips

Epoch 1/30


100%|██████████| 100/100 [24:48<00:00, 14.89s/it]


Train Loss: 1.3887
Train Accuracy: 24.75%


100%|██████████| 10/10 [03:13<00:00, 19.39s/it]


Val Loss: 1.3864
Val Accuracy: 25.00%

Epoch 2/30


...

Val Loss: 1.9969
Val Accuracy: 27.50%

Epoch 30/30


100%|██████████| 100/100 [12:41<00:00,  7.61s/it]


Train Loss: 0.7195
Train Accuracy: 67.12%


100%|██████████| 10/10 [01:11<00:00,  7.15s/it]

Val Loss: 1.9970
Val Accuracy: 27.50%
